# Aula 12 · Lab 1 — Corpus Próprio + 3 Técnicas RAG Integradas

> **Objetivo:** Construir o pipeline RAG do seu projeto final integrando pelo menos 3 técnicas estudadas no curso.

## Roteiro

1. Carregar e preparar seu corpus (PDF/DOCX/texto)
2. Implementar Técnica 1: Chunking semântico por artigo/seção
3. Implementar Técnica 2: Query Expansion com LLM
4. Implementar Técnica 3: Cross-Encoder Reranking
5. Integrar as 3 técnicas no pipeline final
6. Teste de sanidade: 5 perguntas sobre seu corpus

---
**Proporção:** 10% teoria / 90% prática  
**Referência:** Gao et al. (2023). RAG Survey. arXiv:2312.10997.

In [ ]:
# PASSO 0: Instalação
!pip install -q openai langchain langchain-openai langfuse ragas \
    opensearch-py sentence-transformers docling python-dotenv datasets nest-asyncio
print("✅ Pronto")

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "sk-..."  # ← SUBSTITUA

# Identifique seu corpus
# Opções: 'juridico', 'seguranca_publica', 'medico', 'academico', 'custom'
TEMA_CORPUS = "juridico"  # ← ALTERE para o seu tema
print(f"Tema do corpus: {TEMA_CORPUS}")

---
## Teoria (10%): Por que combinar 3 técnicas?

Cada técnica RAG resolve um problema diferente:

- **Chunking por artigo/seção**: Documentos jurídicos têm estrutura natural (Art. 1º, Art. 2º...). Respeitar essa estrutura evita cortar o contexto no meio de uma norma.
- **Query Expansion**: O usuário pode perguntar "multa por vazamento de dados" mas o texto da lei diz "sanção administrativa por descumprimento". Expandir a query com sinônimos melhora o recall.
- **Cross-Encoder Reranking**: O bi-encoder (embedding) é rápido mas impreciso. O cross-encoder reavalia os top-K com maior precisão, priorizando os chunks mais relevantes.

**Resultado:** Melhor recall (mais resultados relevantes) + melhor precision (menos ruído no contexto).

In [ ]:
# ============================================================
# PASSO 1: Carregue seu corpus
# ============================================================
# Opção A: Upload de arquivo no Colab
# from google.colab import files
# uploaded = files.upload()
# caminho = list(uploaded.keys())[0]

# Opção B: Texto direto (para teste)
# Substitua pelo conteúdo real do seu corpus
SEU_CORPUS = [
    {
        "id": "meu_doc_001",
        "titulo": "[SUBSTITUA: Título do seu documento]",
        "texto": """[SUBSTITUA: Cole aqui o texto do seu primeiro documento.
        Pode ser uma lei, uma portaria, um acórdão, um regulamento, etc.
        Quanto mais texto, melhor será a avaliação.
        Recomendado: mínimo de 500 palavras por documento.]""",
        "tipo": "[SUBSTITUA: legislacao / jurisprudencia / doutrina / normativa]",
        "fonte": "[SUBSTITUA: Ex: Diário Oficial, 01/01/2024]",
    },
    # Adicione mais documentos aqui (mínimo 5 para o projeto final)
]

print(f"📚 Corpus carregado: {len(SEU_CORPUS)} documentos")
print("⚠️  Lembre-se: substitua os placeholders pelo seu conteúdo real!")

In [ ]:
# ============================================================
# TÉCNICA 1: Chunking Semântico por Artigo/Seção
# Ideal para: legislação, normativas, contratos
# ============================================================

import re
from typing import List, Dict, Any

def chunking_por_artigo(documento: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Divide o documento respeitando a estrutura de artigos.
    Padrão reconhecido: 'Art. N', 'Art.N', '§ N', 'Artigo N'
    Para outros tipos de documento, adapte os padrões regex.
    """
    texto = documento["texto"]

    # Padrão para artigos de leis brasileiras
    # Adicione outros padrões conforme seu corpus
    padrao = r'(?=Art\.?\s*\d+|Artigo\s+\d+|§\s*\d+|CAPÍTULO|SEÇÃO|TÍTULO)'

    partes = re.split(padrao, texto, flags=re.IGNORECASE)
    partes = [p.strip() for p in partes if len(p.strip()) > 50]  # Remove fragmentos

    # Se não encontrou padrão de artigos, usa chunking por sentença
    if len(partes) <= 1:
        from langchain.text_splitter import RecursiveCharacterTextSplitter
        splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64)
        partes = splitter.split_text(texto)
        print(f"   ℹ️  Padrão de artigos não encontrado — usando chunking recursivo")

    chunks = []
    for i, parte in enumerate(partes):
        chunks.append({
            "chunk_id":     f"{documento['id']}_art_{i:03d}",
            "doc_id":       documento["id"],
            "titulo":       documento["titulo"],
            "texto":        parte,
            "tipo":         documento.get("tipo", "desconhecido"),
            "fonte":        documento.get("fonte", ""),
            "chunk_index":  i,
            "tecnica":      "chunking_por_artigo",
        })

    return chunks


# Aplica ao corpus
todos_chunks = []
for doc in SEU_CORPUS:
    chunks_doc = chunking_por_artigo(doc)
    todos_chunks.extend(chunks_doc)
    print(f"  [{doc['id']}] → {len(chunks_doc)} chunks")

print(f"\n✅ Técnica 1 concluída: {len(todos_chunks)} chunks totais")

In [ ]:
# ============================================================
# TÉCNICA 2: Query Expansion com LLM
# Gera variações da pergunta para ampliar o recall
# ============================================================

from openai import OpenAI
import json

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


def expandir_query(
    pergunta: str,
    n_variacoes: int = 3,
    dominio: str = "direito brasileiro"
) -> List[str]:
    """
    Gera N variações da pergunta usando LLM.
    Técnica: Multi-query retrieval — mais queries = mais cobertura.
    """
    prompt = f"""Você é um especialista em {dominio}.
Gere {n_variacoes} variações da pergunta abaixo, usando diferentes terminologias 
técnicas e jurídicas que cobrem o mesmo tema.
Retorne SOMENTE uma lista JSON de strings, sem explicações.

Pergunta original: {pergunta}

Resposta (lista JSON):"""

    resposta = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=300
    )

    texto = resposta.choices[0].message.content.strip()

    try:
        # Extrai JSON da resposta
        inicio = texto.find("[")
        fim    = texto.rfind("]") + 1
        variacoes = json.loads(texto[inicio:fim])
    except Exception:
        # Fallback: divide por quebra de linha
        variacoes = [l.strip().strip('"') for l in texto.split("\n") if l.strip()]

    # Inclui a pergunta original + variações
    return [pergunta] + variacoes[:n_variacoes]


# Teste
PERGUNTA_TESTE = "Qual é a punição por vazamento de dados pessoais?"
queries_expandidas = expandir_query(
    PERGUNTA_TESTE,
    n_variacoes=3,
    dominio="direito digital e LGPD"
)

print(f"✅ Técnica 2 — Query Expansion")
print(f"\nPergunta original: {PERGUNTA_TESTE}")
print("\nVariações geradas:")
for i, q in enumerate(queries_expandidas):
    print(f"  {i}. {q}")

In [ ]:
# ============================================================
# TÉCNICA 3: Cross-Encoder Reranking
# Reordena os chunks recuperados com maior precisão
# ============================================================

from sentence_transformers import CrossEncoder

# Carrega modelo de reranking (multilíngue — funciona bem com PT)
# Alternativa mais leve: 'cross-encoder/ms-marco-MiniLM-L-6-v2'
print("Carregando modelo de reranking (pode demorar no primeiro uso)...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2")
print("✅ Modelo carregado")


def reranker_chunks(
    pergunta: str,
    chunks: List[Dict[str, Any]],
    top_n: int = 5
) -> List[Dict[str, Any]]:
    """
    Reavalia a relevância de cada chunk em relação à pergunta.
    O cross-encoder considera o par (pergunta, chunk) juntos — mais preciso.
    """
    if not chunks:
        return []

    # Cria pares (pergunta, texto_do_chunk) para o cross-encoder
    pares = [(pergunta, c["texto"][:512]) for c in chunks]  # Limita a 512 chars

    scores = reranker.predict(pares)

    # Adiciona score ao chunk e ordena
    chunks_com_score = [
        {**chunk, "rerank_score": float(score)}
        for chunk, score in zip(chunks, scores)
    ]

    return sorted(chunks_com_score, key=lambda x: x["rerank_score"], reverse=True)[:top_n]


# Teste de reranking com os chunks do corpus
if todos_chunks:
    resultado_rerank = reranker_chunks(
        pergunta=PERGUNTA_TESTE,
        chunks=todos_chunks[:20],  # Simula top-20 do retrieval
        top_n=5
    )
    print(f"\n✅ Técnica 3 — Cross-Encoder Reranking")
    print(f"\nTop 5 chunks após reranking para: '{PERGUNTA_TESTE}'")
    for i, chunk in enumerate(resultado_rerank, 1):
        print(f"\n  [{i}] Score: {chunk['rerank_score']:.4f}")
        print(f"       Fonte: {chunk['titulo'][:60]}")
        print(f"       Texto: {chunk['texto'][:100]}...")

In [ ]:
# ============================================================
# PIPELINE FINAL: INTEGRAÇÃO DAS 3 TÉCNICAS
# ============================================================

import numpy as np

def gerar_embedding(texto: str) -> List[float]:
    """Gera embedding de um texto."""
    r = openai_client.embeddings.create(input=[texto], model="text-embedding-3-small")
    return r.data[0].embedding


def similaridade_cosseno(v1: List[float], v2: List[float]) -> float:
    """Calcula similaridade de cosseno entre dois vetores."""
    a, b = np.array(v1), np.array(v2)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))


def pipeline_rag_3_tecnicas(
    pergunta: str,
    chunks_corpus: List[Dict],
    embeddings_corpus: List[List[float]] = None,
    top_k_retrieval: int = 15,
    top_n_final: int = 5,
    verbose: bool = True
) -> Dict[str, Any]:
    """
    Pipeline completo com 3 técnicas integradas:
    1. Chunking por artigo (aplicado na ingestão)
    2. Query Expansion
    3. Cross-Encoder Reranking
    """
    import time
    inicio = time.time()

    if not chunks_corpus:
        return {"erro": "Corpus vazio. Adicione documentos ao SEU_CORPUS."}

    # TÉCNICA 2: Expande a query
    if verbose: print("🔍 Expandindo query...")
    queries = expandir_query(pergunta, n_variacoes=2)

    # Gera embeddings de todas as queries expandidas
    embeddings_queries = [gerar_embedding(q) for q in queries]

    # Gera embeddings do corpus (se não fornecidos)
    if embeddings_corpus is None:
        if verbose: print("🔍 Gerando embeddings do corpus (uma vez)...")
        embeddings_corpus = [gerar_embedding(c["texto"][:512]) for c in chunks_corpus]

    # Retrieval: busca por todas as queries expandidas
    if verbose: print("🔍 Recuperando chunks...")
    scores_por_chunk: Dict[int, float] = {}

    for emb_query in embeddings_queries:
        for i, emb_chunk in enumerate(embeddings_corpus):
            score = similaridade_cosseno(emb_query, emb_chunk)
            # Mantém o maior score entre as queries expandidas
            scores_por_chunk[i] = max(scores_por_chunk.get(i, 0.0), score)

    # Seleciona top-K para o reranker
    top_indices = sorted(scores_por_chunk, key=scores_por_chunk.get, reverse=True)[:top_k_retrieval]
    candidatos = [chunks_corpus[i] for i in top_indices]

    # TÉCNICA 3: Reranking cross-encoder
    if verbose: print("🔍 Aplicando reranking...")
    chunks_finais = reranker_chunks(pergunta, candidatos, top_n=top_n_final)

    # Gera resposta
    if verbose: print("🔍 Gerando resposta...")
    contexto = "\n\n---\n\n".join([
        f"[Fonte: {c['titulo']}]\n{c['texto']}"
        for c in chunks_finais
    ])

    resposta = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Responda com base EXCLUSIVAMENTE nos documentos fornecidos. Cite a fonte."},
            {"role": "user", "content": f"Contexto:\n{contexto}\n\nPergunta: {pergunta}"}
        ],
        temperature=0.0, max_tokens=800
    )

    latencia = round(time.time() - inicio, 2)

    return {
        "pergunta": pergunta,
        "queries_expandidas": queries,
        "resposta": resposta.choices[0].message.content,
        "chunks_usados": len(chunks_finais),
        "fontes": [c["titulo"] for c in chunks_finais],
        "latencia_s": latencia,
        "tokens": resposta.usage.total_tokens,
    }


# Teste final
resultado = pipeline_rag_3_tecnicas(
    pergunta=PERGUNTA_TESTE,
    chunks_corpus=todos_chunks
)

print("\n" + "="*60)
print("RESULTADO DO PIPELINE RAG (3 TÉCNICAS)")
print("="*60)
print(f"Pergunta: {resultado['pergunta']}")
print(f"Queries expandidas: {resultado['queries_expandidas']}")
print(f"\nResposta:\n{resultado['resposta']}")
print(f"\nFontes: {resultado['fontes']}")
print(f"Latência: {resultado['latencia_s']}s | Tokens: {resultado['tokens']}")

---
## Exercício — Teste de Sanidade

Antes de avançar para os labs seguintes, valide seu pipeline com **5 perguntas** sobre seu corpus.  
Anote as respostas e avalie manualmente:

| # | Pergunta | Resposta OK? | Fonte Citada? | Observação |
|---|----------|:------------:|:-------------:|------------|
| 1 | | ☐ | ☐ | |
| 2 | | ☐ | ☐ | |
| 3 | | ☐ | ☐ | |
| 4 | | ☐ | ☐ | |
| 5 | | ☐ | ☐ | |

Se a acurácia for < 60%, revise o corpus e o chunking antes de avançar.

In [ ]:
# Cole aqui suas 5 perguntas de teste
PERGUNTAS_SANIDADE = [
    "[Pergunta 1 sobre seu corpus]",
    "[Pergunta 2 sobre seu corpus]",
    "[Pergunta 3 sobre seu corpus]",
    "[Pergunta 4 sobre seu corpus]",
    "[Pergunta 5 sobre seu corpus]",
]

for i, pergunta in enumerate(PERGUNTAS_SANIDADE, 1):
    print(f"\n{'='*50}")
    print(f"Pergunta {i}: {pergunta}")
    r = pipeline_rag_3_tecnicas(pergunta, todos_chunks, verbose=False)
    print(f"Resposta: {r['resposta'][:200]}...")
    print(f"Fontes: {r['fontes'][:2]}")